# 01 — Pre-processing Pipeline
**BraTS 2023 GLI Brain Tumor Segmentation | RCOEM 2026–27**

This notebook:
1. Scans the dataset directory and lists all patient folders
2. Loads each patient's 4 MRI modalities (FLAIR, T1, T1ce, T2)
3. Normalises each modality (Z-score within brain mask)
4. Saves preprocessed volumes as compressed `.npz` files (fast to load during training)
5. Generates a preprocessing report with statistics

**⚠️ Only one line to change:** set `DATASET_PATH` to match your DGX folder.

**Estimated time on DGX:** ~1.5 hours for the full dataset

## ⚙️ Configuration — Change This Before Running

In [ ]:
import sys
import os

# ─────────────────────────────────────────────────────────
# ⬇️  CHANGE THIS PATH to wherever you extracted BraTS 2023
DATASET_PATH      = "/home/yourname/BraTS2023_Training_Data"   # ← EDIT ME
PREPROCESSED_PATH = "/home/yourname/BraTS2023_Preprocessed"    # ← output folder

MAX_PATIENTS = 600    # Use None to process all, or an int for a subset
QUICK_TEST   = False  # Set True to test on 5 patients first
# ─────────────────────────────────────────────────────────

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
print(f'Dataset path    : {DATASET_PATH}')
print(f'Output path     : {PREPROCESSED_PATH}')
print(f'Max patients    : {MAX_PATIENTS}')
print(f'Quick test mode : {QUICK_TEST}')

## 📦 Imports

In [ ]:
import re
import time
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict

from src.dataset import get_patient_folders, get_file_paths, load_volume, normalize_volume, remap_labels_brats2023

print('✅ All imports successful')

## 1️⃣ Discover Patient Folders

In [ ]:
all_folders = get_patient_folders(DATASET_PATH)

if QUICK_TEST:
    all_folders = all_folders[:5]
    print(f'[QUICK TEST] Using {len(all_folders)} patients')
elif MAX_PATIENTS:
    all_folders = all_folders[:MAX_PATIENTS]

print(f'\n📂 Found {len(all_folders)} patient folders')
print(f'   First: {Path(all_folders[0]).name}')
print(f'   Last:  {Path(all_folders[-1]).name}')

# Show first 3 folder contents
print('\n📁 Sample folder contents (first patient):')
for f in sorted(Path(all_folders[0]).iterdir()):
    print(f'   {f.name}')

## 2️⃣ Check One Patient (Sanity Check)

In [ ]:
# Load first patient and check shapes + stats
test_folder = all_folders[0]
paths = get_file_paths(test_folder)

print(f'Patient: {Path(test_folder).name}\n')
for key, path in paths.items():
    vol = load_volume(path)
    print(f'  {key:8s} | shape: {vol.shape} | min: {vol.min():.1f} | max: {vol.max():.1f} | dtype: {vol.dtype}')

## 3️⃣ Verify Label Distribution (First Patient)

In [ ]:
seg_vol = remap_labels_brats2023(load_volume(paths['seg']))
unique, counts = np.unique(seg_vol, return_counts=True)
total_voxels = seg_vol.size

label_names = {0: 'Background', 1: 'NCR (Necrotic Core)', 2: 'ED (Edema)', 3: 'ET (Enhancing)'}
print('Label distribution (first patient):')
print(f'{"Label":>8} {"Name":25} {"Count":>10} {"Percent":>10}')
print('-' * 60)
for u, c in zip(unique, counts):
    print(f'{int(u):>8} {label_names.get(int(u), "Unknown"):25} {c:>10,} {100*c/total_voxels:>9.2f}%')

## 4️⃣ Visualise Raw MRI + Labels (First Patient)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(f'Patient: {Path(test_folder).name} — Axial mid-slice', fontsize=14, fontweight='bold')

modality_keys = ['flair', 't1', 't1ce', 't2']
modality_titles = ['FLAIR', 'T1 Native', 'T1 CE', 'T2']
cmap_modality = 'gray'

mid_z = seg_vol.shape[2] // 2

# Row 1: raw modalities
for i, (key, title) in enumerate(zip(modality_keys, modality_titles)):
    vol = load_volume(paths[key])
    axes[0, i].imshow(vol[:, :, mid_z].T, cmap=cmap_modality, origin='lower')
    axes[0, i].set_title(title, fontsize=12)
    axes[0, i].axis('off')

# Row 1 col 5: segmentation label map
axes[0, 4].imshow(seg_vol[:, :, mid_z].T, cmap='tab10', origin='lower', vmin=0, vmax=3)
axes[0, 4].set_title('GT Labels', fontsize=12)
axes[0, 4].axis('off')

# Row 2: normalised modalities
brain_mask = load_volume(paths['flair']) > 0
for i, (key, title) in enumerate(zip(modality_keys, modality_titles)):
    vol_raw  = load_volume(paths[key])
    vol_norm = normalize_volume(vol_raw, brain_mask)
    axes[1, i].imshow(vol_norm[:, :, mid_z].T, cmap=cmap_modality, origin='lower')
    axes[1, i].set_title(f'{title} (normalised)', fontsize=10)
    axes[1, i].axis('off')

# Row 2 col 5: label overlay on FLAIR
flair_norm = normalize_volume(load_volume(paths['flair']), brain_mask)
axes[1, 4].imshow(flair_norm[:, :, mid_z].T, cmap='gray', origin='lower')
mask_overlay = np.ma.masked_where(seg_vol[:, :, mid_z] == 0, seg_vol[:, :, mid_z])
axes[1, 4].imshow(mask_overlay.T, cmap='tab10', alpha=0.5, origin='lower', vmin=0, vmax=3)
axes[1, 4].set_title('FLAIR + Labels', fontsize=10)
axes[1, 4].axis('off')

plt.tight_layout()
plt.savefig('sample_patient_visualisation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_patient_visualisation.png')

## 5️⃣ Run Preprocessing Pipeline (All Patients)

In [ ]:
stats_records = []
failed = []
start_time = time.time()

for folder in tqdm(all_folders, desc='Preprocessing'):
    patient_id = Path(folder).name
    out_path   = Path(PREPROCESSED_PATH) / f'{patient_id}.npz'

    # Skip if already preprocessed (allows resuming after interruption)
    if out_path.exists():
        continue

    try:
        paths = get_file_paths(folder)

        # Build brain mask from FLAIR
        flair_raw   = load_volume(paths['flair'])
        brain_mask  = flair_raw > 0

        # Load, normalise all 4 modalities
        modality_vols = []
        for key in ['flair', 't1', 't1ce', 't2']:
            vol  = load_volume(paths[key])
            norm = normalize_volume(vol, brain_mask)
            modality_vols.append(norm)

        image = np.stack(modality_vols, axis=0).astype(np.float32)  # (4, H, W, D)
        seg   = remap_labels_brats2023(load_volume(paths['seg']))   # (H, W, D)

        # Save as compressed numpy archive
        np.savez_compressed(out_path, image=image, seg=seg)

        # Record stats
        unique, counts = np.unique(seg, return_counts=True)
        label_counts = dict(zip(unique.tolist(), counts.tolist()))
        stats_records.append({
            'patient_id': patient_id,
            'image_shape': str(image.shape),
            'ncr_voxels':  label_counts.get(1, 0),
            'ed_voxels':   label_counts.get(2, 0),
            'et_voxels':   label_counts.get(3, 0),
            'total_tumor': sum([label_counts.get(i, 0) for i in [1,2,3]]),
        })

    except Exception as e:
        failed.append({'patient_id': patient_id, 'error': str(e)})
        print(f'\n  ❌ Failed: {patient_id} — {e}')

elapsed = time.time() - start_time
print(f'\n✅ Preprocessing complete!')
print(f'   Processed : {len(stats_records)} patients')
print(f'   Failed    : {len(failed)} patients')
print(f'   Time      : {elapsed/60:.1f} minutes')

## 6️⃣ Save Preprocessing Report

In [ ]:
if stats_records:
    df = pd.DataFrame(stats_records)
    report_path = Path(PREPROCESSED_PATH) / 'preprocessing_report.csv'
    df.to_csv(report_path, index=False)
    print(f'📊 Report saved: {report_path}')
    print(f'\nDataset statistics:')
    print(df[['ncr_voxels','ed_voxels','et_voxels','total_tumor']].describe().round(0))

    # Histogram of tumor sizes
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col, title, color in zip(axes,
        ['ncr_voxels','ed_voxels','et_voxels'],
        ['NCR (Necrotic Core)','ED (Edema)','ET (Enhancing)'],
        ['#e74c3c','#3498db','#2ecc71']):
        ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
        ax.set_title(f'{title}\nMedian: {df[col].median():.0f} voxels')
        ax.set_xlabel('Voxel count')
        ax.set_ylabel('# Patients')
        ax.grid(True, alpha=0.3)
    plt.suptitle('Tumor Sub-Region Size Distributions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('tumor_size_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

if failed:
    print(f'\n⚠️ {len(failed)} patients failed:')
    for f in failed:
        print(f'  {f["patient_id"]}: {f["error"]}')

print('\n🏁 Notebook 01 complete. Proceed to 02_explore_data.ipynb')